# FREUID 2026 — CNN (Kaggle Notebook)

Classificação binária de documentos: `0` legítimo · `1` fraude.

**Setup:** Add Input → competition `the-freuid-challenge-2026-ijcai-ecai`  
**GPU:** Settings → Accelerator → GPU T4/P100  

Não publica imagens do dataset (licença Microblink).

In [ ]:
!pip -q install timm
import os, json, random
from pathlib import Path
import numpy as np
import pandas as pd
import torch
import torch.nn as nn
from torch.utils.data import Dataset, DataLoader
from torchvision import transforms
from PIL import Image
from sklearn.model_selection import GroupShuffleSplit
from sklearn.metrics import roc_auc_score, roc_curve, classification_report
import timm
from tqdm.auto import tqdm

ROOT = Path('/kaggle/input/the-freuid-challenge-2026-ijcai-ecai')
TRAIN_DIR = ROOT / 'train' / 'train'
LABELS = ROOT / 'train_labels.csv'
OUT = Path('/kaggle/working')
SEED = 42
IMG_SIZE = 384
BATCH = 32
EPOCHS = 5
MAX_TRAIN = 12000  # ajuste conforme GPU/tempo

random.seed(SEED); np.random.seed(SEED); torch.manual_seed(SEED)
device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
print('device', device, 'train_dir', TRAIN_DIR.exists())

In [ ]:
def freuid_score(y_true, y_score):
    auc = roc_auc_score(y_true, y_score)
    fpr, tpr, _ = roc_curve(y_true, y_score)
    mask = fpr <= 0.01
    tpr_at_1 = tpr[mask].max() if mask.any() else 0.0
    if auc + tpr_at_1 == 0:
        return 1.0, auc, 1 - tpr_at_1
    score = 1 - 2 * auc * tpr_at_1 / (auc + tpr_at_1)
    return float(score), float(auc), float(1 - tpr_at_1)

df = pd.read_csv(LABELS)
df['file'] = df['image_path'].map(lambda p: TRAIN_DIR / Path(p).name)
df = df[df['file'].map(lambda p: p.exists())].reset_index(drop=True)
print(len(df), df['label'].mean())

# subset estratificado para caber no tempo de GPU
if len(df) > MAX_TRAIN:
    rare = df[df['is_digital'] == False]
    rest = df[df['is_digital'] == True]
    n = MAX_TRAIN - len(rare)
    rest = rest.groupby(['label','type'], group_keys=False).apply(
        lambda g: g.sample(min(len(g), max(1, n // (df.groupby(['label','type']).ngroups))), random_state=SEED)
    )
    df = pd.concat([rare, rest]).drop_duplicates('id').sample(frac=1, random_state=SEED).reset_index(drop=True)
    if len(df) > MAX_TRAIN:
        df = df.sample(MAX_TRAIN, random_state=SEED).reset_index(drop=True)
print('usando', len(df))

In [ ]:
types = sorted(df['type'].unique())
heldout = types[-1]
train_pool = df[df['type'] != heldout].reset_index(drop=True)
ood = df[df['type'] == heldout].reset_index(drop=True)
gss = GroupShuffleSplit(n_splits=1, test_size=0.2, random_state=SEED)
tr, va = next(gss.split(train_pool, train_pool['label'], groups=train_pool['type']))
train_df, val_df = train_pool.iloc[tr].reset_index(drop=True), train_pool.iloc[va].reset_index(drop=True)
print('heldout', heldout, 'train', len(train_df), 'val', len(val_df), 'ood', len(ood))

train_tf = transforms.Compose([
    transforms.Resize((IMG_SIZE, IMG_SIZE)),
    transforms.RandomHorizontalFlip(p=0.0),  # docs: evitar flip que quebra layout
    transforms.ColorJitter(0.1, 0.1, 0.1, 0.05),
    transforms.GaussianBlur(kernel_size=3, sigma=(0.1, 1.0)),
    transforms.ToTensor(),
    transforms.Normalize([0.485,0.456,0.406],[0.229,0.224,0.225]),
])
eval_tf = transforms.Compose([
    transforms.Resize((IMG_SIZE, IMG_SIZE)),
    transforms.ToTensor(),
    transforms.Normalize([0.485,0.456,0.406],[0.229,0.224,0.225]),
])

class DocDS(Dataset):
    def __init__(self, frame, tf):
        self.frame = frame
        self.tf = tf
    def __len__(self):
        return len(self.frame)
    def __getitem__(self, i):
        r = self.frame.iloc[i]
        img = Image.open(r['file']).convert('RGB')
        x = self.tf(img)
        y = torch.tensor(int(r['label']), dtype=torch.float32)
        return x, y

train_loader = DataLoader(DocDS(train_df, train_tf), batch_size=BATCH, shuffle=True, num_workers=2, pin_memory=True)
val_loader = DataLoader(DocDS(val_df, eval_tf), batch_size=BATCH, shuffle=False, num_workers=2)
ood_loader = DataLoader(DocDS(ood, eval_tf), batch_size=BATCH, shuffle=False, num_workers=2)

In [ ]:
model = timm.create_model('efficientnet_b0', pretrained=True, num_classes=1)
model = model.to(device)
opt = torch.optim.AdamW(model.parameters(), lr=1e-4, weight_decay=1e-4)
crit = nn.BCEWithLogitsLoss()

def run_epoch(loader, train=True):
    model.train(train)
    total, ys, ss = 0.0, [], []
    ctx = torch.enable_grad() if train else torch.no_grad()
    with ctx:
        for x, y in tqdm(loader, leave=False):
            x, y = x.to(device), y.to(device)
            logits = model(x).squeeze(1)
            loss = crit(logits, y)
            if train:
                opt.zero_grad(); loss.backward(); opt.step()
            total += loss.item() * len(y)
            ys.append(y.detach().cpu().numpy())
            ss.append(torch.sigmoid(logits).detach().cpu().numpy())
    y = np.concatenate(ys); s = np.concatenate(ss)
    score, auc, apcer = freuid_score(y, s)
    return total/len(loader.dataset), auc, apcer, score, y, s

history = []
best = 1e9
for ep in range(1, EPOCHS+1):
    tr = run_epoch(train_loader, True)
    va = run_epoch(val_loader, False)
    row = {'epoch': ep, 'train_loss': tr[0], 'val_auc': va[1], 'val_apcer': va[2], 'val_freuid': va[3]}
    history.append(row)
    print(row)
    if va[3] < best:
        best = va[3]
        torch.save(model.state_dict(), OUT/'best_efficientnet_b0.pt')

model.load_state_dict(torch.load(OUT/'best_efficientnet_b0.pt', map_location=device))
ood_m = run_epoch(ood_loader, False)
print('OOD', heldout, {'auc': ood_m[1], 'apcer@1': ood_m[2], 'freuid': ood_m[3]})
pd.DataFrame(history).to_csv(OUT/'cnn_history.csv', index=False)
json.dump({'heldout': heldout, 'val_best_freuid': best, 'ood_auc': ood_m[1], 'ood_freuid': ood_m[3]}, open(OUT/'cnn_metrics.json','w'), indent=2)
print('OK — salvos em /kaggle/working')